# Checkpoint II · Examen blanc — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/checkpoints/partie_2/03_examen_solutions.ipynb)

Les solutions des deux questions de code (CP2.10 et CP2.11), exécutées, puis l'enregistrement des réponses que vérifie la partie B du notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`). Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Partie A · Les deux questions de code

In [2]:
# Data and tools of the two code questions (run this cell first)
import numpy as np

california = wb.datasets.load_california()                    # 20 640 districts (1990 census)
FEATURES = california.columns[:8].tolist()                    # MedInc, HouseAge, ..., Latitude, Longitude
rows = np.sort(np.random.default_rng(2610).choice(len(california), 2000, replace=False))
X = california[FEATURES].to_numpy(dtype=float)[rows]          # shape (2000, 8)
y = california["MedHouseVal"].to_numpy(dtype=float)[rows]     # median house value (hundreds of thousands of $)


def make_cells(size):
    """Id of the square cell (side `size`, in degrees) of the latitude-longitude grid that holds each district."""
    row = np.floor(np.round(X[:, 6] / size, 8)).astype(int)    # latitude band
    col = np.floor(np.round(X[:, 7] / size, 8)).astype(int)    # longitude band
    return row * 100_000 + col


CELLS = {size: make_cells(size) for size in (0.5, 0.1, 0.05)}  # the three grids of CP2.10 d)


def cell_means(cells, y):
    """Dict: cell id -> mean of y over the districts of that cell (only the districts passed)."""
    return {cell: float(y[cells == cell].mean()) for cell in np.unique(cells)}


def lookup(cells, means, default):
    """For each district, the mean of its cell in `means`, or `default` if its cell is not in `means`."""
    return np.array([means.get(cell, default) for cell in cells], dtype=float)


def kfold(n, k=5, seed=0):
    """List of k pairs (train_indices, val_indices): the n indices, shuffled once, cut into k folds."""
    folds = np.array_split(np.random.default_rng(seed).permutation(n), k)
    return [(np.concatenate([folds[j] for j in range(k) if j != i]), folds[i]) for i in range(k)]


def ridge_fit(F, y, alpha):
    """Ridge regression with an intercept that is not penalized: returns (w, b)."""
    F_mean, y_mean = F.mean(axis=0), y.mean()
    Fc = F - F_mean
    w = np.linalg.solve(Fc.T @ Fc + alpha * np.eye(F.shape[1]), Fc.T @ (y - y_mean))
    return w, y_mean - F_mean @ w


def ridge_predict(model, F):
    w, b = model
    return F @ w + b


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


print("X:", X.shape, "· y:", y.shape, "· cells of 0.1°:", len(np.unique(CELLS[0.1])))

X: (2000, 8) · y: (2000,) · cells of 0.1°: 519


### CP2.10 — La fuite cachée d'une validation croisée 🐛 ★ ⏱️ 9 min · 1,5 point
**Objectif :** trouver ce qu'une validation croisée a appris de ses folds de validation, et la corriger.

Un collègue estime la RMSE d'une régression Ridge par validation croisée à 5 folds, sur les 2 000 districts de `X` et `y`. En plus des huit features, il ajoute « le prix du quartier » : la grille latitude-longitude est découpée en cases de 0,1° de côté (`CELLS[0.1]` donne la case de chaque district), et chaque district reçoit la moyenne des prix (`y`) des districts de sa case. Il standardise ensuite les neuf colonnes, puis lance la validation croisée. Exécute son code (cellule suivante).

a) Deux étapes de son code apprennent quelque chose des données avant la boucle de validation croisée. Lesquelles ? Laquelle fausse vraiment son score, et pourquoi l'autre ne le change presque pas ? (sur ta feuille) **[0,4 pt]**
b) Écris `cv_rmse_fixed(X, y, cells, k=5, alpha=1.0)`, qui fait le même calcul **sans fuite**. Dans chaque tour (`kfold(len(y), k)`, comme chez le collègue) : le prix du quartier de chaque district, d'entraînement comme de validation, est la moyenne des prix des districts **d'entraînement du tour** qui sont dans sa case ; un district dont la case ne contient aucun district d'entraînement reçoit la moyenne des prix d'entraînement du tour ; la standardisation utilise la moyenne et l'écart-type des lignes d'entraînement du tour. La fonction renvoie la moyenne des $k$ RMSE de validation, sans modifier ses arguments. **[0,6 pt]**
c) La RMSE moyenne de ta fonction avec les cases de 0,1° (3 décimales, sur ta feuille). **[0,2 pt]**
d) Le collègue a aussi comparé des cases de 0,5°, 0,1° et 0,05° avec son code (sortie de sa cellule), et garde les plus petites. Compare avec ta fonction : quelle taille garderais-tu ? Pourquoi la fuite le poussait-elle vers les plus petites cases ? (sur ta feuille) **[0,3 pt]**

In [3]:
# The colleague's code (do not change this cell: run it and read what it prints)
def cv_rmse_colleague(X, y, cells, k=5, alpha=1.0):
    neighbourhood = lookup(cells, cell_means(cells, y), y.mean())   # "price of the neighbourhood" of each district
    F = np.column_stack([X, neighbourhood])
    F = (F - F.mean(axis=0)) / F.std(axis=0)                         # standardize the 9 columns
    scores = []
    for train, val in kfold(len(y), k):
        model = ridge_fit(F[train], y[train], alpha)
        scores.append(rmse(y[val], ridge_predict(model, F[val])))
    return float(np.mean(scores))


for size, cells in CELLS.items():
    print(f"cells of {size}°: the colleague's cross-validated RMSE = {cv_rmse_colleague(X, y, cells):.4f}")

cells of 0.5°: the colleague's cross-validated RMSE = 0.7032
cells of 0.1°: the colleague's cross-validated RMSE = 0.5256
cells of 0.05°: the colleague's cross-validated RMSE = 0.3998


In [4]:
# a) Two steps learn from ALL the districts before the folds exist: the cell means of the target (each
#    district's own price is in the mean of its cell: the validation targets leak into the features), and the
#    standardization (a mean and a std over 2 000 districts barely move when 400 of them are removed).
def cv_rmse_fixed(X, y, cells, k=5, alpha=1.0):
    """Mean validation RMSE of the colleague's model over k folds, without leakage (see CP2.10 b)."""
    scores = []
    for train, val in kfold(len(y), k):
        means = cell_means(cells[train], y[train])                    # learnt on the training districts only
        neighbourhood = lookup(cells, means, y[train].mean())          # for every district, train and validation
        F = np.column_stack([X, neighbourhood])
        F = (F - F[train].mean(axis=0)) / F[train].std(axis=0)        # standardization fitted on the training rows
        model = ridge_fit(F[train], y[train], alpha)
        scores.append(rmse(y[val], ridge_predict(model, F[val])))
    return float(np.mean(scores))


fixed_10 = {size: cv_rmse_fixed(X, y, cells) for size, cells in CELLS.items()}
leaky_10 = {size: cv_rmse_colleague(X, y, cells) for size, cells in CELLS.items()}
for size in CELLS:
    print(f"cells of {size}°: colleague {leaky_10[size]:.4f} · without leakage {fixed_10[size]:.4f}")

cells of 0.5°: colleague 0.7032 · without leakage 0.7163
cells of 0.1°: colleague 0.5256 · without leakage 0.6811
cells of 0.05°: colleague 0.3998 · without leakage 0.7285


In [5]:
# Classic wrong procedures of b), each recorded with its message
def variant_10(cells, *, means="train", default="train"):
    """The cross-validated RMSE when the neighbourhood feature is computed another (wrong) way."""
    scores = []
    for train, val in kfold(len(y), 5):
        fallback = y[train].mean() if default == "train" else y.mean()
        if means == "all":                                     # the colleague's means, over all the districts
            neighbourhood = lookup(cells, cell_means(cells, y), fallback)
        else:
            neighbourhood = lookup(cells, cell_means(cells[train], y[train]), fallback)
        if means == "own_fold":                                # validation districts encoded with their own fold
            neighbourhood[val] = lookup(cells[val], cell_means(cells[val], y[val]), y[val].mean())
        if means == "leave_one_out":                           # training districts encoded without their own price
            for i in train:
                others = train[(cells[train] == cells[i]) & (train != i)]
                neighbourhood[i] = y[others].mean() if others.size else y[train].mean()
        F = np.column_stack([X, neighbourhood])
        F = (F - F[train].mean(axis=0)) / F[train].std(axis=0)
        scores.append(rmse(y[val], ridge_predict(ridge_fit(F[train], y[train], 1.0), F[val])))
    return float(np.mean(scores))


def no_neighbourhood():
    """The cross-validated RMSE of the 8 features alone (the new feature forgotten)."""
    scores = []
    for train, val in kfold(len(y), 5):
        F = (X - X[train].mean(axis=0)) / X[train].std(axis=0)
        scores.append(rmse(y[val], ridge_predict(ridge_fit(F[train], y[train], 1.0), F[val])))
    return float(np.mean(scores))


cells_10 = CELLS[0.1]
wb.record("CP2.10c", fixed_10[0.1], decimals=4, mistakes={
    "c'est encore la fuite du collègue : les moyennes par case utilisent les prix de TOUS les districts, ceux du fold de validation compris":
    variant_10(cells_10, means="all"),
    "les districts de validation reçoivent la moyenne de leur propre fold : leur prix entre encore dans leur feature ; prends les moyennes des seuls districts d'entraînement du tour":
    variant_10(cells_10, means="own_fold"),
    "ta version retire aussi son propre prix à chaque district d'entraînement : c'est une bonne idée, mais l'énoncé demande, pour tous les districts, la moyenne des districts d'entraînement de la case":
    variant_10(cells_10, means="leave_one_out"),
    "ta fonction oublie le prix du quartier : le modèle doit garder les neuf colonnes du collègue":
    no_neighbourhood()})
wb.record("CP2.10d", [fixed_10[size] for size in CELLS], decimals=4, mistakes={
    "c'est encore la fuite du collègue : les moyennes par case utilisent les prix de TOUS les districts, ceux du fold de validation compris":
    [variant_10(cells, means="all") for cells in CELLS.values()],
    "les districts de validation reçoivent la moyenne de leur propre fold : leur prix entre encore dans leur feature ; prends les moyennes des seuls districts d'entraînement du tour":
    [variant_10(cells, means="own_fold") for cells in CELLS.values()],
    "pour une case sans district d'entraînement, le défaut est la moyenne des prix d'entraînement du tour, pas celle de tous les districts":
    [variant_10(cells, default="all") for cells in CELLS.values()]})

WB_ANSWER {"decimals": 4, "hash": "9eb2331ac08b322714b2900631bf0e5be3fb2a1833f7967c1def3cefb4f202b0", "hash_coarse": "b51882bbc4f63e1d7254d96d1394b32e6a6afff1b91e2f94c6b33ae1de90536d", "id": "CP2.10c", "kind": "float", "magnitude": -1, "mistakes": {"51292bc8546ce80d67f161c44d0b508a5b549d56e2090b9dd400d8a821488bd9": "ta fonction oublie le prix du quartier : le modèle doit garder les neuf colonnes du collègue", "8b4af72a7b11879aaf3066e7d165a58deedf16add7b4c51d8968d8c66ba15dc6": "c'est encore la fuite du collègue : les moyennes par case utilisent les prix de TOUS les districts, ceux du fold de validation compris", "9963ce6ae7c2d3ea686f8ba9162d9796c7c82084a0e257ef6c7f33739e2128f7": "ta version retire aussi son propre prix à chaque district d'entraînement : c'est une bonne idée, mais l'énoncé demande, pour tous les districts, la moyenne des districts d'entraînement de la case", "cafd3558f96ef752b5ff585869a3ca3998a22ba50e92f8884bb9766b4136d2d7": "les districts de validation reçoivent la moye

WB_ANSWER {"decimals": 4, "element_coarse": [["bd178a98278ee8708e6b39f10d0204e8a7d0231c08bbde6fb80097f2db9c98b2"], ["5a8ccc34aaaed44c49031d7e5fb5e9a6fe76ed9581cc693eac6b74a3a923fc29"], ["120f8b2eeb2a0f6c0ef6e6140c4e07479cd715feed3e592ec6fb44a50a854dd6"]], "element_hashes": ["471fa66d0d45e072455689b71f7c2e4aa71b17103836abf978f5ea7763574177", "7fea980bb9e88c1e4aacc986c6f4836c77e07da9719f8b22105ba79b182509e6", "57b5e67214ef525d92f7d7d8d3981bab3179c95c580e330543710d23b446b513"], "hash": "c6fb9a13adc0126fda7e6828fe478ef5a01e06f63f145e2fb4531f87d5ff4c17", "id": "CP2.10d", "kind": "array", "mistakes": {"110deebba7b51d5e3ae8637c8581bdb66b4f277296d915c6107830a44c77cab4": "pour une case sans district d'entraînement, le défaut est la moyenne des prix d'entraînement du tour, pas celle de tous les districts", "76399b1f4b9fd676cbef64daac5d0aa2708f69a5d81b125a8b31b9409be722f7": "les districts de validation reçoivent la moyenne de leur propre fold : leur prix entre encore dans leur feature ; prends le

### CP2.11 — Coder `epsilon_greedy_action` et une moyenne incrémentale 🔨 ★ ⏱️ 9 min · 1 point
**Objectif :** programmer, sans `mylearn`, les deux briques d'un agent de bandit.

a) Écris `epsilon_greedy_action(q_values, epsilon, rng)`. Avec la probabilité $\varepsilon$ (un tirage `rng.random() < epsilon`), elle explore : un bras au hasard parmi **tous** les bras, `rng.integers(K)` ($K$ = nombre de bras). Sinon, elle exploite : un bras d'estimation maximale, tiré au hasard parmi les ex aequo avec `rng.choice`. Elle renvoie un `int` et ne modifie pas `q_values`. **[0,5 pt]**
b) Écris `incremental_estimates(rewards, step=None)`, qui renvoie la liste des estimations successives d'un bras, une après chaque récompense, en partant de $Q = 0$ : à la $n$-ième récompense $R$, $Q \leftarrow Q + a\,(R - Q)$, avec $a = 1/n$ si `step` vaut `None` (la moyenne exacte), et $a$ = `step` sinon (un pas constant). La fonction ne recalcule jamais la somme de toutes les récompenses. **[0,3 pt]**
c) Avec ta fonction, la dernière estimation pour les récompenses `[1, 0, 0, 1, 1, 1, 0, 1]` et un pas constant de 0,25 (4 décimales, sur ta feuille). **[0,2 pt]**

In [6]:
def epsilon_greedy_action(q_values, epsilon, rng):
    """The arm chosen by the epsilon-greedy rule (an int), see CP2.11 a)."""
    q_values = np.asarray(q_values, dtype=float)
    if rng.random() < epsilon:                                   # explore: any arm, the best one included
        return int(rng.integers(len(q_values)))
    best = np.flatnonzero(q_values == q_values.max())            # all the arms of maximal estimate
    return int(rng.choice(best))                                 # ties broken at random


def incremental_estimates(rewards, step=None):
    """The list of the successive estimates Q, one after each reward, starting from Q = 0 (CP2.11 b)."""
    q, estimates = 0.0, []
    for n, reward in enumerate(rewards, start=1):
        a = 1 / n if step is None else step
        q = q + a * (reward - q)
        estimates.append(q)
    return estimates


rewards_11 = [1, 0, 0, 1, 1, 1, 0, 1]
print("exact means   :", np.round(incremental_estimates(rewards_11), 4).tolist())
print("step of 0.25  :", np.round(incremental_estimates(rewards_11, step=0.25), 4).tolist())

exact means   : [1.0, 0.5, 0.3333, 0.5, 0.6, 0.6667, 0.5714, 0.625]
step of 0.25  : [0.25, 0.1875, 0.1406, 0.3555, 0.5166, 0.6375, 0.4781, 0.6086]


In [7]:
def constant_step(rewards, step, q):
    for reward in rewards:
        q = q + step * (reward - q)
    return q


wb.record("CP2.11c", incremental_estimates(rewards_11, step=0.25)[-1], decimals=4,
          mistakes={"c'est la moyenne exacte des récompenses : avec un pas constant, a = step à chaque récompense, pas 1/n":
                    float(np.mean(rewards_11)),
                    "l'énoncé part de Q = 0 : la première récompense ne remplace pas l'estimation, elle la déplace de a (R − Q)":
                    constant_step(rewards_11[1:], 0.25, float(rewards_11[0])),
                    "relis la mise à jour : Q ← Q + a (R − Q), où (R − Q) est l'écart entre la récompense et l'estimation":
                    constant_step(rewards_11, 0.75, 0.0)})

WB_ANSWER {"decimals": 4, "hash": "b92e53e23f7b44ae793c68cd9bdbf2e92bb80f2e4ad2c724678d8e08a5f52a9f", "hash_coarse": "0dc72d6e45223992ae19e674c638a4562ef386899b36794cfa845cfbe6f87ab5", "id": "CP2.11c", "kind": "float", "magnitude": -1, "mistakes": {"6c4d489f7cfb449834a6c4fc651df87bc4d5b105cada5ec2f88000fe17f9d5fd": "l'énoncé part de Q = 0 : la première récompense ne remplace pas l'estimation, elle la déplace de a (R − Q)", "e1d65292b35e5fdd0de70962171f3bec44de60ad63ee73405c2271df2157e828": "relis la mise à jour : Q ← Q + a (R − Q), où (R − Q) est l'écart entre la récompense et l'estimation", "f7cce7aed02c2f8ca1f2ad664bc1929142d0c80cfb933113dfdc79ad83ca6172": "c'est la moyenne exacte des récompenses : avec un pas constant, a = step à chaque récompense, pas 1/n"}, "sign": 1}
📝 Ex CP2.11c : réponse enregistrée (float, 4 décimale(s)).


## Partie B · Les réponses vérifiées après l'examen

Chaque réponse est calculée à partir des nombres du sujet ; les erreurs classiques reçoivent un message qui dit où chercher.

In [8]:
# The paper questions only use the numbers of their statements (01_examen_sujet.md)
import math
from fractions import Fraction

import numpy as np


def ovo_votes(winners, classes="ABCDE"):
    """Votes of a one-versus-one from the winners of the duels, listed in the order of the statement."""
    return [winners.count(c) for c in classes]


WINNERS_2A = "ACAECBEDCD"                   # A-B, A-C, A-D, A-E, B-C, B-D, B-E, C-D, C-E, D-E
WINNERS_2B = "BCDABDBCED"
POINTS_3 = np.array([[2, 6], [3, 6], [4, 6], [6, 4], [9, 5], [9, 6]], dtype=float)


def assign_3(centres):
    d2 = ((POINTS_3[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2)
    return d2.argmin(axis=1), d2.min(axis=1)


CENTRES0_3 = POINTS_3[[1, 2]]                                   # c1 = P2, c2 = P3
LABELS1_3, D2_0_3 = assign_3(CENTRES0_3)
CENTRES1_3 = np.array([POINTS_3[LABELS1_3 == k].mean(axis=0) for k in range(2)])
LABELS2_3, _ = assign_3(CENTRES1_3)
CENTRES2_3 = np.array([POINTS_3[LABELS2_3 == k].mean(axis=0) for k in range(2)])
LABELS3_3, D2_2_3 = assign_3(CENTRES2_3)
D2_PP_3 = ((POINTS_3 - POINTS_3[0]) ** 2).sum(axis=1)             # k-means++ from P1


def r_orange(d):
    return 2 * math.sqrt(d) - 1                                   # balloons centred at (±2, ..., ±2), radius 1


N_5 = 333
N_TEST_5 = math.ceil(0.2 * N_5)
REST_5 = N_5 - N_TEST_5
SE_5 = math.sqrt(0.94 * 0.06 / N_TEST_5)
x_6 = np.array([1, 2, 3, 4, 5], dtype=float)
y_6 = np.array([2, 3, 5, 4, 6], dtype=float)


def ridge_1d(x, y, lam):
    sxx = ((x - x.mean()) ** 2).sum()
    sxy = ((x - x.mean()) * (y - y.mean())).sum()
    w = sxy / (sxx + lam)
    return w, y.mean() - w * x.mean()


def sse_6(x, y, lam):
    w, b = ridge_1d(x, y, lam)
    return float(((y - w * x - b) ** 2).sum())


X_8 = [(0, 0), (0, 1), (1, 0), (1, 1)]
Y_8 = [1, 1, 1, -1]                                               # NAND, 0 coded -1


def perceptron_epoch(w, b, labels=Y_8, strict=False):
    """One epoch of the perceptron rule of the fiche; returns (w, b, the values of z, the number of corrections)."""
    w, zs, corrections = list(w), [], 0
    for (x1, x2), label in zip(X_8, labels):
        z = w[0] * x1 + w[1] * x2 + b
        zs.append(z)
        if (label * z < 0) if strict else (label * z <= 0):
            w, b, corrections = [w[0] + label * x1, w[1] + label * x2], b + label, corrections + 1
    return w, b, zs, corrections


W1_8, B1_8, Z1_8, C1_8 = perceptron_epoch([0, 0], 0)
W2_8, B2_8, _, _ = perceptron_epoch(W1_8, B1_8)
TP_13, FP_13 = 120, 40
FN_13, TN_13 = 150 - TP_13, 1000 - 150 - FP_13
P_13, R_13 = TP_13 / (TP_13 + FP_13), TP_13 / (TP_13 + FN_13)
FPR_13 = FP_13 / (FP_13 + TN_13)


def entropy_bits(counts):
    p = np.asarray(counts, dtype=float) / sum(counts)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

**CP2.1 — Vrai ou faux justifiés (le verdict seulement ; les justifications se notent avec le corrigé)**

In [9]:
wb.record("CP2.1a", False)
wb.record("CP2.1b", False)
wb.record("CP2.1c", False)
wb.record("CP2.1d", False)
wb.record("CP2.1e", True)
wb.record("CP2.1f", False)
wb.record("CP2.1g", True)
wb.record("CP2.1h", True)

WB_ANSWER {"hash": "23308b7aa3d629827f0d023ea4dfbd8924168e6ded7ce6b7d89b1ecd2da05ee9", "id": "CP2.1a", "kind": "bool"}
📝 Ex CP2.1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "49025e069ca808ce55d42f9ce20c1dae32329c1739f8500f94a9888621e0d6a1", "id": "CP2.1b", "kind": "bool"}
📝 Ex CP2.1b : réponse enregistrée (bool).
WB_ANSWER {"hash": "9638dbc3ccc243c173b7cb17e5c56e12f84bc00f8dd9d0d325de4931acec9914", "id": "CP2.1c", "kind": "bool"}
📝 Ex CP2.1c : réponse enregistrée (bool).
WB_ANSWER {"hash": "0626a0042a3f66c030ca15369aa75e58ead10c70e533121746a119a5d0083e42", "id": "CP2.1d", "kind": "bool"}
📝 Ex CP2.1d : réponse enregistrée (bool).
WB_ANSWER {"hash": "af474697f11dd8f5a3c4b61a564c1edca27002518edb6ab8d7181348ce2f5180", "id": "CP2.1e", "kind": "bool"}
📝 Ex CP2.1e : réponse enregistrée (bool).
WB_ANSWER {"hash": "4e2d3e6ab6404edf027916ed11b3c0c3c2bc05cdab97574469cf20c45dde411a", "id": "CP2.1f", "kind": "bool"}
📝 Ex CP2.1f : réponse enregistrée (bool).
WB_ANSWER {"hash": "af1433016b0d54

**CP2.2 — Un-contre-tous, un-contre-un**

In [10]:
wb.record("CP2.2a", [12, 12 * 11 // 2], mistakes={"chaque paire de classes ne fait qu'un duel : divise le nombre de couples ordonnés par 2": [12, 132],
                     "K² compte aussi des « duels » d'une classe contre elle-même : un duel oppose deux classes différentes": [12, 144],
                     "l'ordre demandé est [N_OvR, N_OvO]": [66, 12]})
wb.record("CP2.2b", min(k for k in range(2, 100) if k * (k - 1) // 2 >= 100), mistakes={"le nombre de duels est K(K − 1)/2, pas K²": 10,
                     "chaque paire de classes ne fait qu'un duel : K(K − 1)/2 duels, pas K(K − 1)": 11,
                     "calcule K(K − 1)/2 pour ta réponse : atteint-il 100 ?": 14})
wb.record("CP2.2c", ovo_votes(WINNERS_2A))
wb.record("CP2.2d", "C")
wb.record("CP2.2e", ovo_votes(WINNERS_2B))
wb.record("CP2.2f", "B", mistakes={"relis la règle d'égalité de mylearn : elle départage par l'indice des classes, sans regarder les duels": "D"})
wb.record("CP2.2g", "D", mistakes={"ici, la règle départage les deux classes à égalité par le duel qui les a opposées, pas par leur indice": "B"})

WB_ANSWER {"decimals": 0, "element_hashes": ["2693f05ba9acd5470411aaa704e8326dd84333e5727a83a3d8f49162ebbba316", "c2cbe07cf8c5783e3e94712f1ae185e2a54dd0b498efcc2ff889b103aa7a11ff"], "hash": "7f6bbc43b6c42f357a06d34c6ad351b530943650518b10cb45519571b372ac8c", "id": "CP2.2a", "integer": true, "kind": "array", "mistakes": {"77a70d5d2d90642a4fed7b8e3bf0ff5f17ae76b624765a2ba1491448ee0079ec": "chaque paire de classes ne fait qu'un duel : divise le nombre de couples ordonnés par 2", "8881213fb609cd233ecfc6a73e0a892183b219a8faec676ae0986c1c231b2ab6": "l'ordre demandé est [N_OvR, N_OvO]", "9010b7c3c9286440e57953403eff0df01e11ce1a6ea17a5934d24bc09ba19585": "K² compte aussi des « duels » d'une classe contre elle-même : un duel oppose deux classes différentes"}, "shape": [2]}
📝 Ex CP2.2a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"hash": "74bbd5ac6b960c6652fda70706c2b6ab31ac29b370a8bc328ee2e5e50f28364e", "id": "CP2.2b", "kind": "int", "magnitude": 1, "mistakes": {"95d2cf5f398a8bc91d53

**CP2.3 — Une itération de k-means et l'inertie obtenue**

In [11]:
wb.record("CP2.3a", (LABELS1_3 + 1).tolist(), mistakes={"compare les distances au carré de P3 aux deux centres de DÉPART (c1 = P2, c2 = P3)": [1, 1, 1, 2, 2, 2],
                     "les numéros des clusters suivent ceux des centres : le cluster 1 est celui de c1 = P2": [2, 2, 1, 1, 1, 1],
                     "les clusters sont numérotés 1 et 2, comme leurs centres c1 et c2 (pas 0 et 1)": [0, 0, 1, 1, 1, 1]})
wb.record("CP2.3b", float(D2_0_3.sum()), decimals=2, mistakes={"l'inertie additionne des distances AU CARRÉ": float(np.sqrt(D2_0_3).sum())})
wb.record("CP2.3c", CENTRES1_3.tolist(), decimals=2, mistakes={"l'ordre demandé est [[x1, y1], [x2, y2]] : le centre du cluster 1 d'abord": CENTRES1_3[::-1].tolist()})
wb.record("CP2.3d", float(((POINTS_3 - CENTRES1_3[LABELS1_3]) ** 2).sum()), decimals=2, mistakes={"l'inertie additionne des distances AU CARRÉ": float(np.sqrt(((POINTS_3 - CENTRES1_3[LABELS1_3]) ** 2).sum(axis=1)).sum()),
                     "d) garde l'affectation de a), mais mesure les distances aux NOUVEAUX centres de c), pas aux centres de départ": float(D2_0_3.sum())})
wb.record("CP2.3e", int(np.flatnonzero(LABELS1_3 != LABELS2_3)[0] + 1), mistakes={f"compare les distances au carré de chaque point aux deux NOUVEAUX centres de c) : P{point} reste du même côté": point
                     for point in range(1, 7) if point != int(np.flatnonzero(LABELS1_3 != LABELS2_3)[0] + 1)})
wb.record("CP2.3f", CENTRES2_3.tolist(), decimals=2, mistakes={"l'ordre demandé est [[x1, y1], [x2, y2]] : le centre du cluster 1 d'abord": CENTRES2_3[::-1].tolist()})
wb.record("CP2.3g", float(D2_2_3.sum()), decimals=2)
wb.record("CP2.3h", bool((LABELS3_3 != LABELS2_3).any()))
wb.record("CP2.3i", float(D2_PP_3[3:].sum() / D2_PP_3.sum()), decimals=3, mistakes={"k-means++ tire chaque point avec une probabilité proportionnelle au CARRÉ de sa distance D(x)": float(np.sqrt(D2_PP_3[3:]).sum() / np.sqrt(D2_PP_3).sum()),
                     "les points ne sont pas tirés uniformément : chacun a une probabilité proportionnelle à D(x)²": 0.6,
                     "le premier centre, P1, a D(P1) = 0 : il ne peut pas être tiré une seconde fois": 0.5,
                     "la question porte sur l'un des TROIS points P4, P5, P6": float(D2_PP_3[4] / D2_PP_3.sum())})

WB_ANSWER {"decimals": 0, "element_hashes": ["7dbe8d45aec8ea3b23db7be3ffc658b154dc342a73c6f7922f010489e00fc4e9", "95ef44e1a309a4eaead978875952a700f9a1f1646ab8c43901e165fabf3c91a2", "9855a73cd7c04f9a608bf34254d857f407acb18844456dcadc8fcd76d706ff34", "aee36856260f1303db53850ef76556b026e79bb374f86563bb55d4fbae41dbdb", "c5e2be58ced95201ecf96f72b0fd60c8fad7bc8d8825618abc3d073ba1edd931", "c1b9b72518d10cb4c85a3aeb65d3c668a85cef5bfbadd798432e79553dd1d3a6"], "hash": "fba1f616520847565e456b8e8780db6357aa5e8aee6351498b02aade87c5424e", "id": "CP2.3a", "integer": true, "kind": "array", "mistakes": {"64c705bdc94c8ac1d6070779678b95c20b0f3223ccb5d3e601516c0fbb5fcc0b": "compare les distances au carré de P3 aux deux centres de DÉPART (c1 = P2, c2 = P3)", "81e5d81a3161088eb65a4411dcb79c1d042daea13bb01bb647300ef78a666b42": "les numéros des clusters suivent ceux des centres : le cluster 1 est celui de c1 = P2", "cb2fd733921eef5a6accc6999a66e316bfb64a228bf73f4407a2c6342c1a58df": "les clusters sont numérotés

**CP2.4 — Densité d'échantillons et hyper-orange**

In [12]:
wb.record("CP2.4a", [2000 / 4 ** 3, 2000 / 4 ** 6], decimals=3, mistakes={"en dimension d, il y a b^d cases (b par axe), pas b × d": [2000 / 12, 2000 / 24]})
wb.record("CP2.4b", 2 * 4 ** 8, mistakes={"c'est le nombre de cases : multiplie-le par la densité voulue": 4 ** 8,
                     "le nombre de cases est b^d : la base est le nombre de cases par axe, l'exposant la dimension": 2 * 8 ** 4})
wb.record("CP2.4c", min(d for d in range(1, 30) if 2000 / 4 ** d < 0.01), mistakes={"calcule la densité dans cette dimension : elle n'est pas encore sous 0,01": 8})
wb.record("CP2.4d", r_orange(9), decimals=3, mistakes={"le centre d'un ballon a toutes ses coordonnées égales à ±2 : sa distance à l'origine vaut 2√d, pas √d": math.sqrt(9) - 1,
                     "l'orange touche les ballons : son rayon est la distance au centre d'un ballon MOINS le rayon du ballon": 2 * math.sqrt(9)})
wb.record("CP2.4e", min(d for d in range(1, 30) if r_orange(d) > 3), mistakes={"dans cette dimension, l'orange touche les faces de la boîte sans en sortir : elle sort quand son rayon DÉPASSE la demi-largeur 3": 4,
                     "c'est la dimension où sortait l'orange de la fiche (boîte de côté 4, ballons centrés en ±1) : ici, la boîte a pour côté 6 et les ballons sont centrés en ±2": 10})
wb.record("CP2.4f", 1 - 0.95 ** 30, decimals=3, mistakes={"c'est la part du volume à l'INTÉRIEUR de la boule de rayon 0,95 : la peau, c'est le reste": 0.95 ** 30,
                     "une peau de 5 % du rayon laisse une boule intérieure de rayon 0,95": 1 - 0.9 ** 30})

WB_ANSWER {"decimals": 3, "element_coarse": [["24ded53c18589a564737fb929be39cd1baa2295c610360457af9dedf478925af"], ["1a75a22a1993caf494893798d9de1a880437e19a426f1e776b7909c1948a0561"]], "element_hashes": ["c42b7067aadb7b6aca0850c73dc0ce449294d14e66a959f46a773f691d2e0153", "4728fc70ec41b8d1c2ca5371e1020bbf6f159681be4f7b6f1589df002c5d8b61"], "hash": "e44f956d3e9195051584f8a97438faf768213cb2ed049e094aedd41977825066", "id": "CP2.4a", "kind": "array", "mistakes": {"a3c01ce7da3b4619e080b44ac73a18715865be2c8694f42b595b114b0d5a1315": "en dimension d, il y a b^d cases (b par axe), pas b × d"}, "shape": [2]}
📝 Ex CP2.4a : réponse enregistrée (array, 3 décimale(s)).
WB_ANSWER {"hash": "8428701ad538c3d84705c420dcbbc02e797ff64e112f2a80ba0d6ef9832ade43", "id": "CP2.4b", "kind": "int", "magnitude": 5, "mistakes": {"3bf008ef81196703a5d11a8a45b1cf8d5a9a8f71835144a56aa077d1776807c2": "le nombre de cases est b^d : la base est le nombre de cases par axe, l'exposant la dimension", "f6c531f8e8040057ee03841e

**CP2.5 — Plan d'évaluation**

In [13]:
wb.record("CP2.5a", N_TEST_5, mistakes={"la taille du test s'arrondit vers le HAUT : ⌈t·n⌉": 66})
wb.record("CP2.5b", [REST_5 // 5 + (i < REST_5 % 5) for i in range(5)], mistakes={"les n mod k PREMIERS folds reçoivent l'exemple de plus": [53, 53, 53, 53, 54],
                     "la validation croisée porte sur les manchots qui RESTENT une fois le test mis de côté": [67, 67, 67, 66, 66]})
wb.record("CP2.5c", 15 * 5 + 1, mistakes={"n'oublie pas le réentraînement final du réglage retenu": 75,
                     "chaque réglage demande un entraînement PAR FOLD": 16})
wb.record("CP2.5d", 5 * (15 * 5 + 1), mistakes={"dans chaque tour extérieur, le réglage retenu est réentraîné une fois sur la partie qui reste": 375,
                     "l'énoncé ne prévoit pas de réentraînement après les cinq tours extérieurs": 381,
                     "chaque tour extérieur fait UNE recherche de c), pas une par fold intérieur": 5 * 5 * (15 * 5 + 1),
                     "d) ne compte que les cinq tours extérieurs : la recherche finale qui produirait le modèle livré n'en fait pas partie": 5 * (15 * 5 + 1) + 15 * 5 + 1})
wb.record("CP2.5e", [SE_5, 1.96 * SE_5], decimals=3, mistakes={"n est la taille du jeu de TEST, sur lequel l'accuracy a été mesurée": [math.sqrt(0.94 * 0.06 / N_5), 1.96 * math.sqrt(0.94 * 0.06 / N_5)],
                     "la demi-largeur de l'intervalle à 95 % utilise 1,96, pas 2": [SE_5, 2 * SE_5]})
wb.record("CP2.5f", [1, 2, 5], ordered=False, mistakes={"il manque une fuite : relis les protocoles qui calculent quelque chose sur des données avant de les découper": [1, 5],
                     "il manque une fuite : un prétraitement fait avant le tirage même du test compte aussi": [2, 5],
                     "il manque une fuite : un choix fait en regardant le test est aussi une fuite": [1, 2],
                     "un protocole de trop : un prétraitement ajusté sur l'entraînement du tour seulement n'est pas une fuite": [1, 2, 3, 5],
                     "un protocole de trop : évaluer une seule fois sur le test, après le choix, est la bonne pratique": [1, 2, 4, 5]})

WB_ANSWER {"hash": "45c7d714df892b2473159937a338b7ce38826f4cf406464d9f381c7dbddcab5d", "id": "CP2.5a", "kind": "int", "magnitude": 1, "mistakes": {"5cad93c63fcb9eb0071d972ca2be99c9eaedb4683b2acbf7b773b7817d560b1a": "la taille du test s'arrondit vers le HAUT : ⌈t·n⌉"}, "sign": 1}
📝 Ex CP2.5a : réponse enregistrée (int).
WB_ANSWER {"decimals": 0, "element_hashes": ["8b1bc4bbbf8d55022a9766a744cbe852ccfc6ba74ca22520473ab4f1a529adfe", "6bfa6731acd68d043cb1f074f4f3937e67eacf7c93682f6b8564d0a3c77d4baf", "1b5e4b1e283f5395d74709394f1c67942980c1a47333d5903dd58e024a9d01ec", "669779536f683152bf5a26d93d122ad256a3cae7d6509cc7726c3700f8b4e17a", "d2f30eae530750dc5432229784c38311784da4dfe8f1d0cfc7db144b6358577b"], "hash": "9b76e1acca53c1178cd6df26dd38ffc2d18890b68af80c1efd012d87ca51dfb7", "id": "CP2.5b", "integer": true, "kind": "array", "mistakes": {"0dd8c992d756c3f572ea9f23b809bcd7274d5cd0f0afcccd9855eb11dde07582": "la validation croisée porte sur les manchots qui RESTENT une fois le test mis de côté

**CP2.6 — Ridge en dimension 1**

In [14]:
wb.record("CP2.6a", list(ridge_1d(x_6, y_6, 0)), decimals=3, mistakes={"le modèle a une ordonnée à l'origine : b = ȳ − w x̄": [ridge_1d(x_6, y_6, 0)[0], 0.0],
                     "l'ordre demandé est [w, b]": list(ridge_1d(x_6, y_6, 0))[::-1],
                     "w* utilise les écarts aux moyennes : S_xy et S_xx": [float(x_6 @ y_6 / (x_6 @ x_6)), float(y_6.mean() - x_6 @ y_6 / (x_6 @ x_6) * x_6.mean())]})
wb.record("CP2.6b", list(ridge_1d(x_6, y_6, 5)), decimals=3, mistakes={"b dépend de w : recalcule b = ȳ − w x̄ avec le nouveau w": [ridge_1d(x_6, y_6, 5)[0], ridge_1d(x_6, y_6, 0)[1]],
                     "w* utilise S_xy et S_xx, calculés sur les écarts aux moyennes": [float(x_6 @ y_6 / (x_6 @ x_6 + 5)), float(y_6.mean() - x_6 @ y_6 / (x_6 @ x_6 + 5) * x_6.mean())]})
wb.record("CP2.6c", [sse_6(x_6, y_6, 0), sse_6(x_6, y_6, 5)], decimals=3, mistakes={"l'ordre demandé est λ = 0, puis λ = 5": [sse_6(x_6, y_6, 5), sse_6(x_6, y_6, 0)],
                     "c) demande les résidus seuls, sans la pénalité λ w²": [sse_6(x_6, y_6, 0), sse_6(x_6, y_6, 5) + 5 * ridge_1d(x_6, y_6, 5)[0] ** 2],
                     "c) demande la SOMME des carrés des résidus, pas leur moyenne": [sse_6(x_6, y_6, 0) / 5, sse_6(x_6, y_6, 5) / 5]})
wb.record("CP2.6d", sum(np.array(ridge_1d(10 * x_6, y_6, 5)) * [50, 1]), decimals=3, mistakes={"les données ont changé d'unité : recalcule S_x'x', S_x'y, w et b avec x' = 10 x": 5.2,
                     "λ = 5 compte encore : w = S_x'y / (S_x'x' + λ), ce n'est pas tout à fait la pente des moindres carrés": 5.8,
                     "l'ordonnée à l'origine change aussi : b' = ȳ − w' x̄'": 2.2 + ridge_1d(10 * x_6, y_6, 5)[0] * 50,
                     "le point x = 5 s'écrit x' = 50 dans la nouvelle unité": sum(np.array(ridge_1d(10 * x_6, y_6, 5)) * [5, 1])})

WB_ANSWER {"decimals": 3, "element_coarse": [["30e06ad04214bd757f9eba10c720f7ed0c1342ff90aa8908e81dd7b714054ab9"], ["1bbf0df23cb630ae8bb075a2ce12fd28a7741745361024b38b69f4facd587e75"]], "element_hashes": ["925dab81bfaa62307fb1e25fd1b7e4549d4c683ce50f0d86957ac7950866ee4c", "0673cff5687a2d0a5ec1cacd3132f0e7fe8dfe44ca91e93bc7c3ef64822a1ea3"], "hash": "160d8bb7f715aa8073ee8b4f26dda50f8c8c1e31a642d2491de009414654cc4c", "id": "CP2.6a", "kind": "array", "mistakes": {"11b828a7fa796b42ac7162e9eb818b377bd9f6a969742ea24c91720240b9e060": "l'ordre demandé est [w, b]", "4fb920025af4e9b90f9728a95dd62adbde06d270cdeffff2c9b09522954d89e4": "le modèle a une ordonnée à l'origine : b = ȳ − w x̄", "6974a8c6055e24f7b3711bd7b15f5dfd6e1be6b4648d9a11074097539045b84b": "w* utilise les écarts aux moyennes : S_xy et S_xx"}, "shape": [2]}
📝 Ex CP2.6a : réponse enregistrée (array, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "element_coarse": [["8bd7125e05b5dbc2f969e5dd68362536a3893a757c6c8642c01ae3c32014a65a"], ["aa0d

**CP2.7 — Trois paires de courbes d'apprentissage (la lecture de c se note avec le corrigé)**

In [15]:
wb.record("CP2.7a", 2, mistakes={"l'underfitting se lit sur le NIVEAU des deux erreurs, comparé au plancher du bruit": 1,
                     "l'underfitting se lit sur le NIVEAU des deux erreurs, comparé au plancher du bruit (relis la fiche du ch. 9, §9.2)": 3})
wb.record("CP2.7b", 3, mistakes={"l'overfitting se lit sur l'ÉCART entre les deux courbes : une erreur d'entraînement nettement plus basse que celle de validation": 1,
                     "l'overfitting se lit sur l'ÉCART entre les deux courbes : une erreur d'entraînement nettement plus basse que celle de validation (relis la fiche du ch. 9, §9.2)": 2})
wb.record("CP2.7d", "B", mistakes={"plus de données réduit la variance ; relis ce que dit le niveau des deux courbes du panneau 2": "A",
                     "une régularisation plus forte réduit encore la capacité du modèle": "C",
                     "l'early stopping limite lui aussi la capacité du modèle": "D"})
wb.record("CP2.7e", False)

WB_ANSWER {"hash": "ab34346ffd522c209ea626172ae79308b10cd2269118ad1934023a0b3d971445", "id": "CP2.7a", "kind": "int", "magnitude": 0, "mistakes": {"008a381312f1e8e28ca9b8dc8079d5060a4700e54b9c8c14ecc1452d7aa30164": "l'underfitting se lit sur le NIVEAU des deux erreurs, comparé au plancher du bruit", "36ddecb04cf03523d9081e44236baacdc1cc7861cb9e9ab8c8c89e919d35a6bd": "l'underfitting se lit sur le NIVEAU des deux erreurs, comparé au plancher du bruit (relis la fiche du ch. 9, §9.2)"}, "sign": 1}
📝 Ex CP2.7a : réponse enregistrée (int).
WB_ANSWER {"hash": "c30c217fe88c5ab7bae77c2ec30215fdeec9002ae14ae2e03fed063ebe8f50f5", "id": "CP2.7b", "kind": "int", "magnitude": 0, "mistakes": {"8978d546515a06436d4ba82fd3b914858a6180cdaaeb5be32f18aeb100105561": "l'overfitting se lit sur l'ÉCART entre les deux courbes : une erreur d'entraînement nettement plus basse que celle de validation (relis la fiche du ch. 9, §9.2)", "a74832b65aa0d0878bdd609a21ad809fb1b22fa3ba299f4a0cfbb5cbbb5a347d": "l'overfittin

**CP2.8 — Perceptron : NAND, puis XOR**

In [16]:
wb.record("CP2.8a", Z1_8, mistakes={"z se calcule avec les poids et le biais tels qu'ils sont APRÈS la correction de l'exemple précédent": [0, 0, 0, 0]})
wb.record("CP2.8b", W1_8 + [B1_8], mistakes={"la sortie 0 se code y = −1 : avec y = 0, l'exemple (1, 1) ne corrige rien": [0, 0, 1],
                     "une somme nulle compte comme une erreur (y z ≤ 0) : avec y z < 0, partis de zéro, les poids ne bougeraient jamais": [0, 0, 0],
                     "l'exemple (1, 1) corrige aussi le biais : b ← b + η y": [-1, -1, 1]})
wb.record("CP2.8c", C1_8, mistakes={"une somme nulle compte comme une erreur : y z ≤ 0": 1,
                     "un exemple bien classé (y z > 0) ne déclenche aucune correction": 4})
wb.record("CP2.8d", sum((1 if W1_8[0] * x1 + W1_8[1] * x2 + B1_8 > 0 else -1) == label for (x1, x2), label in zip(X_8, Y_8)), mistakes={"la prédiction vaut +1 seulement si z est STRICTEMENT positif : applique-la aux quatre entrées": 2,
                     "relis la règle de prédiction de d) : +1 quand z > 0, −1 sinon": 3})
wb.record("CP2.8e", W2_8 + [B2_8], mistakes={"une somme nulle compte comme une erreur (y z ≤ 0) : avec y z < 0, partis de zéro, les poids ne bougeraient jamais": [0, 0, 0],
                     "la deuxième époque repart des poids de b) et passe sur les QUATRE exemples": [-1, 0, 2]})

WB_ANSWER {"decimals": 0, "element_hashes": ["ba036b74b799bc4450abd0a5ab3ce84c1bbeac9c06c6031a33c1c62af5103e03", "465a2d383a71e0435b21e79e59b7fc0c54f7571e9224839def8cd4be43c59b97", "28fec315e85277096889aed4d95ef94b8c53a105154446d2baf9ae7eb70d08a5", "8c40f2a7cc1ec02baacf84453942402ba77ee1c11ed823523663017c93611b63"], "hash": "0b84e892104f287ccbb9d50185ba9c0bd620b92d672435db7afad84cbfb39c29", "id": "CP2.8a", "integer": true, "kind": "array", "mistakes": {"e04e58d85ecab05a1607892f4da4b992d16b526314d365136b39bf1395e1c941": "z se calcule avec les poids et le biais tels qu'ils sont APRÈS la correction de l'exemple précédent"}, "shape": [4]}
📝 Ex CP2.8a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 0, "element_hashes": ["593243c97aad0e33090e6febd76ca841c332c4ff35a157e7ccfc845e7c3cf35e", "6020ac53e9d4d43c68937ef8021c9e77adfe2941ad0e6b897e223f6a25a8511b", "b723bf378b406b30b07b17845d3d9a130d962c688732deafc22a6074fd495eda"], "hash": "b3901a6d7ac306ea59f2b4ba882fdbe6d8c200b7

**CP2.9 — Syllogismes et sophismes**

In [17]:
wb.record("CP2.9a", "NSNVNN", mistakes={"relis le raisonnement 2 : ses deux prémisses sont-elles vraies ?": "NVNVNN",
                     "relis le raisonnement 4 : une forme valide ne suffit pas, vérifie ses prémisses": "NSNSNN",
                     "relis le raisonnement 3 : écris sa forme avec des lettres (si … alors …)": "NSVVNN",
                     "relis le raisonnement 6 : écris sa forme avec des lettres (si … alors …)": "NSNVNV",
                     "relis le raisonnement 1 : la conclusion découle-t-elle vraiment des deux prémisses ?": "VSNVNN"})
wb.record("CP2.9b", "EBCA", mistakes={"relis le raisonnement 1 : il n'a pas la forme « si … alors … »": "ABCA",
                     "relis le raisonnement 3 : sa deuxième prémisse nie-t-elle la condition, ou affirme-t-elle la conséquence ?": "EACA",
                     "relis le raisonnement 5 : quel terme est distribué dans la conclusion sans l'être dans sa prémisse ?": "EBDA",
                     "relis le raisonnement 6 : sa deuxième prémisse nie-t-elle la condition, ou affirme-t-elle la conséquence ?": "EBCB",
                     "relis les raisonnements 3 et 6 : lequel nie la condition, lequel affirme la conséquence ?": "EACB"})
wb.record("CP2.9c", False)

WB_ANSWER {"hash": "a06493e433a720f41bf1ea87b167d8002ecb49af70529bfb02446833f155d862", "id": "CP2.9a", "kind": "str", "mistakes": {"248387fbe46271f3a7aa3aa3f33f48823912d39aeba924d891a709e4afa1c4de": "relis le raisonnement 3 : écris sa forme avec des lettres (si … alors …)", "37fbc7ec4ba32ecb8bfb353de0aeb1b9955175b092d07650df1ed6d94c32c2ca": "relis le raisonnement 4 : une forme valide ne suffit pas, vérifie ses prémisses", "7df9ddc2dae3e3bac769ff7f2e130266a4a92ffd04e79f9d471775c5293718b6": "relis le raisonnement 6 : écris sa forme avec des lettres (si … alors …)", "9a274b1101ed59f3fc006dc66f0c66ad9242d7e6e70a52d7b679b2e794446da3": "relis le raisonnement 2 : ses deux prémisses sont-elles vraies ?", "adc397daad3faa4fd6a308a5112f743b0d68cdf04f375f6b688faea12b986560": "relis le raisonnement 1 : la conclusion découle-t-elle vraiment des deux prémisses ?"}}
📝 Ex CP2.9a : réponse enregistrée (str).
WB_ANSWER {"hash": "101c1501bd8cb9c95bff65c27cbd43ddc4ad0ef3f22d2bd744970d9c6168caf7", "id": "CP

**CP2.13 — Parties antérieures : matrice de confusion, Bayes et entropie**

In [18]:
wb.record("CP2.13a", [TP_13, FP_13, FN_13, TN_13], mistakes={"FP : les e-mails signalés qui ne sont pas des spams ; FN : les spams non signalés": [TP_13, FN_13, FP_13, TN_13],
                     "TN : les e-mails qui ne sont ni des spams ni signalés (les 1 000 e-mails se répartissent dans les quatre cases)": [TP_13, FP_13, FN_13, TN_13 + FP_13]})
wb.record("CP2.13b", [P_13, R_13, 2 * P_13 * R_13 / (P_13 + R_13)], decimals=3, mistakes={"precision = TP / (TP + FP) et recall = TP / (TP + FN)": [R_13, P_13, 2 * P_13 * R_13 / (P_13 + R_13)],
                     "le F1 est la moyenne HARMONIQUE de la precision et du recall": [P_13, R_13, (P_13 + R_13) / 2]})
wb.record("CP2.13c", (TP_13 + TN_13) / 1000, decimals=2, mistakes={"c'est la moyenne du recall et de la spécificité : l'accuracy compte les bonnes réponses parmi les 1 000 e-mails": (R_13 + TN_13 / (TN_13 + FP_13)) / 2})
wb.record("CP2.13d", R_13 * 0.4 / (R_13 * 0.4 + FPR_13 * 0.6), decimals=3, mistakes={"la proportion de spams a changé : recalcule P(signalé) avec 40 % de spams": P_13,
                     "c'est P(signalé | spam) : on demande P(spam | signalé)": R_13,
                     "le taux de faux positifs est FP / (FP + TN) : la part des e-mails normaux signalés à tort": R_13 * 0.4 / (R_13 * 0.4 + (1 - FPR_13) * 0.6),
                     "c'est P(spam et signalé) : divise par P(signalé)": R_13 * 0.4})
wb.record("CP2.13e", entropy_bits([30, 10]), decimals=3, mistakes={"c'est en nats : on demande des bits (log₂)": entropy_bits([30, 10]) * math.log(2),
                     "additionne les termes −p log₂ p des DEUX espèces": -0.75 * math.log2(0.75),
                     "le cluster ne contient que deux espèces, et pas à parts égales": math.log2(3)})
wb.record("CP2.13f", entropy_bits([20, 10, 10]), decimals=3, mistakes={"les trois espèces ne sont pas équiprobables dans ce cluster": math.log2(3),
                     "c'est en nats : on demande des bits (log₂)": entropy_bits([20, 10, 10]) * math.log(2)})

WB_ANSWER {"decimals": 0, "element_hashes": ["9af7f39d2f91f0a3892db57ec41a1d8489caeefda3d0558cc8d9a0740ba54c70", "9c8a42b5870e72e9052f2bfe102fc5a646cde5c5759c2db70d3c40a1e53301a1", "d2ef336017901883c100e2cc9fe430b6cd5fdf7f15d8b880d0577cfc6a01f4db", "6accd4f1ead87d175a278ad75c30062d68436781ba769f354e7917dcfe213e15"], "hash": "5309816cf5ac8044ff3954f3be7e0b410b52ad322dc12416c76621574af2ad64", "id": "CP2.13a", "integer": true, "kind": "array", "mistakes": {"c82697b97fe4df808ca3d935e57fd232841f7655ed755e85cca1069ce9cfe516": "TN : les e-mails qui ne sont ni des spams ni signalés (les 1 000 e-mails se répartissent dans les quatre cases)", "eff460e0ec6186bc266ad16dde857e034b23827c58d6ce3abdd9a44bb4b48f86": "FP : les e-mails signalés qui ne sont pas des spams ; FN : les spams non signalés"}, "shape": [4]}
📝 Ex CP2.13a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 3, "element_coarse": [["29b7ce68e3fcb4d0900b5c2c36005e1e31973ee2c816f72c286c68a0329f4766"], ["fb0644d15d48235